# Data loading, featurization & round-trip

**One-time build → push → cached load.** This notebook builds QM9 and ZINC-250k from source,
runs the vocab-independent **sanitize + round-trip** filter, pushes the surviving
**canonical SMILES + targets** to the Hub as
[`nico8771/qm9_clean`](https://huggingface.co/datasets/nico8771/qm9_clean) /
[`nico8771/zinc_clean`](https://huggingface.co/datasets/nico8771/zinc_clean), then demonstrates
re-loading from the Hub and building a `MoleculeDataset`. Dense graph tensors `(X, E)` are
rebuilt lazily from SMILES — never stored.

- **§1–§6** build from source (`use_cache=False`) and inspect (round-trip, drop stats, target norms).
- **§7** pushes the cleaned datasets to the Hub (needs a write `HF_TOKEN`).
- **§8** re-loads from the Hub (`use_cache=True`, public/token-free) and builds the training dataset.

Downstream (training) just calls `load_qm9()` / `load_zinc()` with `use_cache=True` to pull these.

- **QM9** — neutral SMILES + DFT targets (`mu, alpha, homo, lumo, gap, cv`); round-trip *report-only*.
- **ZINC** — keeps `N+`/`O-`; `charge_aware=True`; round-trip filter *applied*; targets (`logP, qed, SAS`) recomputed via RDKit.

## Setup

In [3]:
import os

REPO = "flow-matching-molecules"
if not os.path.isdir(REPO):
    !git clone https://github.com/Nico-Conti/flow-matching-molecules.git
os.chdir(REPO if os.path.basename(os.getcwd()) != REPO else ".")

!pip install -q uv
!uv pip install --system -q -e .
import sys; sys.path.insert(0, os.path.abspath("src"))  # flat src/ layout
print("cwd:", os.getcwd())

Cloning into 'flow-matching-molecules'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 55 (delta 20), reused 48 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 28.93 KiB | 1.52 MiB/s, done.
Resolving deltas: 100% (20/20), done.
cwd: /content/flow-matching-molecules


## 1 · Load QM9

In [4]:
import numpy as np
from dataset.qm9 import load_qm9
from dataset.zinc import load_zinc
from dataset.torch_dataset import MoleculeDataset, collate_dense

LIMIT = None   # set None for the full datasets

# One-time build from source (use_cache=False forces the sanitize/round-trip pass).
# §7 pushes the result to the Hub; §8 then re-loads it with use_cache=True (no rebuild).
qm9 = load_qm9(local_dir="data", limit=LIMIT, use_cache=False)
print("QM9 :", qm9["ds"].num_rows, "molecules | targets", qm9["targets"])
print("  stats:", qm9["stats"])

/usr/local/lib/python3.12/dist-packages/torch_molecule/generator/molgpt/modeling_molgpt.py:112: SyntaxWarning: invalid escape sequence '\['
  self.pattern = "(\[[^\]]+]|<|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\(|\)|\.|=|#|-|\+|\\\\|\/|:|~|@|\?|>|\*|\$|\%[0-9]{2}|[0-9])"


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


qm9.csv:   0%|          | 0.00/29.9M [00:00<?, ?B/s]

Dataset downloaded and saved to data/qm9.csv
QM9 : 132040 molecules | targets ('mu', 'alpha', 'homo', 'lumo', 'gap', 'cv')
  stats: {'kept': 132040, 'drop_vocab': 1845}


## 2 · Load ZINC-250k

In [ ]:
zinc = load_zinc(local_dir="data", limit=LIMIT, use_cache=False, uncharge=True)
print("ZINC:", zinc["ds"].num_rows, "molecules | targets", zinc["targets"])
print("  vocab:", zinc["atom_vocab"], "| charge_aware:", zinc["charge_aware"])
print("  stats:", zinc["stats"])

## 3 · Featurizer round-trip checks
`smiles -> (X, E) -> smiles` should reproduce the canonical molecule.

In [ ]:
from rdkit import Chem
from dataset.featurize import (smiles_to_tensor, tensor_to_mol,
                               build_mol_partial_charges, QM9_ATOMS, ZINC_ATOMS)

def canon(s):
    return Chem.MolToSmiles(Chem.MolFromSmiles(s), isomericSmiles=False)

def check(s, vocab, charge_aware, partial_charges=False):
    X, E = smiles_to_tensor(s, atom_vocab=vocab, charge_aware=charge_aware)
    if partial_charges:
        m = build_mol_partial_charges(X, E, atom_vocab=vocab)   # DeFoG-style decode
    else:
        m, _ = tensor_to_mol(X, E, atom_vocab=vocab)            # strict decode
    got = Chem.MolToSmiles(m, isomericSmiles=False) if m is not None else None
    print(f"  {s:22s} -> {str(got):22s} match={got == canon(s)}")

print("QM9 (neutral, charge_aware):")
for s in ["CCO", "CC(=O)O", "c1ccncc1", "N#Cc1ccccc1"]:
    check(s, QM9_ATOMS, charge_aware=True)


print("ZINC (9 neutral types; element-only encode -> partial-charge decode):")
for s in ["CC[N+](C)(C)C", "Clc1ccccc1", "c1ccc2ccccc2c1", "c1ccncc1"]:
    check(s, ZINC_ATOMS, charge_aware=False, partial_charges=True)

## 4 · Round-trip drop summary
`kept` = round-trips exactly; `kept_no_roundtrip` = featurized but not gated (QM9);
`drop_*` = excluded (vocab / round-trip / parse).

In [7]:
def show(name, stats):
    counts = {k: v for k, v in stats.items() if k.startswith(("drop_", "kept"))}
    if not counts:                       # loaded from the Hub cache (no drop breakdown)
        print(f"{name}: {stats}")
        return
    n = sum(counts.values())
    kept = counts.get("kept", 0) + counts.get("kept_no_roundtrip", 0)
    print(f"{name}: {kept:,}/{n:,} usable ({kept/n:.2%})")
    for k, v in sorted(counts.items()):
        print(f"    {k:20s} {v:>6,} ({v/n:.2%})")

show("QM9 ", qm9["stats"])
print()
show("ZINC", zinc["stats"])

QM9 : 132,040/133,885 usable (98.62%)
    drop_vocab            1,845 (1.38%)
    kept                 132,040 (98.62%)

ZINC: 247,449/249,455 usable (99.20%)
    drop_roundtrip            1 (0.00%)
    drop_vocab            2,005 (0.80%)
    kept                 247,449 (99.20%)


## 5 · Targets present & per-property normalization stats
Confirms `y` is aligned (one row per usable molecule) and has no missing values,
then reports mean/std per property — the conditioning-normalization stats for training.

In [8]:
def target_report(name, d):
    y = np.asarray(d["ds"]["y"])          # (N, T) — the y column as an array
    assert y.shape[0] == d["ds"].num_rows, f"{name}: y rows {y.shape[0]} != #mols {d['ds'].num_rows}"
    n_nan = int(np.isnan(y).sum())
    print(f"{name}: y {y.shape} | targets {d['targets']} | NaNs {n_nan}")
    for j, t in enumerate(d["targets"]):
        c = y[:, j]
        print(f"    {t:6s} mean={c.mean():+.4f}  std={c.std():.4f}  min={c.min():+.4f}  max={c.max():+.4f}")
    return {t: (float(y[:, j].mean()), float(y[:, j].std())) for j, t in enumerate(d["targets"])}

norm_stats = {"qm9": target_report("QM9 ", qm9), "zinc": target_report("ZINC", zinc)}
norm_stats

QM9 : y (132040, 6) | targets ('mu', 'alpha', 'homo', 'lumo', 'gap', 'cv') | NaNs 0
    mu     mean=+2.6497  std=1.4013  min=+0.0000  max=+9.4219
    alpha  mean=+75.2282  std=8.1766  min=+6.3100  max=+143.5300
    homo   mean=-0.2402  std=0.0218  min=-0.4286  max=-0.1325
    lumo   mean=+0.0114  std=0.0469  min=-0.1750  max=+0.1935
    gap    mean=+0.2516  std=0.0472  min=+0.0376  max=+0.6221
    cv     mean=+31.5946  std=4.0670  min=+6.0020  max=+46.9690
ZINC: y (247449, 3) | targets ('logP', 'qed', 'SAS') | NaNs 0
    logP   mean=+2.4554  std=1.4326  min=-6.8762  max=+8.2521
    qed    mean=+0.7323  std=0.1382  min=+0.1166  max=+0.9484
    SAS    mean=+3.0488  std=0.8340  min=+1.1327  max=+7.2893


{'qm9': {'mu': (2.649686285661133, 1.4013045622183835),
  'alpha': (75.2281835031271, 8.176579162904908),
  'homo': (-0.24017115338420242, 0.02181021148983733),
  'lumo': (0.011447526516488584, 0.046940279713890275),
  'gap': (0.25161856104473374, 0.04720733216889512),
  'cv': (31.59463972097742, 4.067021801436)},
 'zinc': {'logP': (2.4554073266778147, 1.4326330654182462),
  'qed': (0.732323774814275, 0.13821898153252765),
  'SAS': (3.0488094076110115, 0.834040727938675)}}

## 6 · Push cleaned datasets to the Hub

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()                        # your .env with HF_TOKEN (write token)
assert os.environ.get("HF_TOKEN"), "set HF_TOKEN in .env before pushing"

from dataset.qm9 import push_qm9
from dataset.zinc import push_zinc

push_qm9(qm9)                                            # -> nico8771/qm9_clean
# Neutral ZINC -> a NEW repo; do not overwrite the charged nico8771/zinc_clean.
push_zinc(zinc, repo_id="nico8771/zinc_neutral")         # data + dataset card
print("pushed qm9_clean + zinc_neutral")

## 7 · Load from the Hub & build the training dataset

Re-loads `nico8771/qm9_clean` from the Hub (`use_cache=True`, no token), wraps it in
`MoleculeDataset` (lazy `smiles -> (X, E)`), and splits **train/val/test in code** with a fixed
seed — the Hub holds one unsplit corpus, so the split lives here, not in the cached artifact.

In [10]:
from torch.utils.data import DataLoader, random_split
import torch

# Re-load from the Hub (use_cache=True -> pulls nico8771/qm9_clean, public/token-free).
qm9 = load_qm9(use_cache=True)
full = MoleculeDataset.from_loader(qm9)          # lazy smiles -> (X, E)

# We store one unsplit corpus, so train/val/test is split here, in code, with a fixed seed.
n = len(full); n_test = int(0.1 * n); n_val = int(0.1 * n)
g = torch.Generator().manual_seed(0)
train, val, test = random_split(full, [n - n_val - n_test, n_val, n_test], generator=g)
print(f"loaded {n:,} from Hub | split train={len(train):,} val={len(val):,} test={len(test):,}")

loader = DataLoader(train, batch_size=4, shuffle=True, collate_fn=collate_dense)
print({k: tuple(v.shape) for k, v in next(iter(loader)).items()})

README.md:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.94M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/132040 [00:00<?, ? examples/s]

loaded 132,040 from Hub | split train=105,632 val=13,204 test=13,204
{'X': (4, 9, 4), 'E': (4, 9, 9, 4), 'y': (4, 6), 'mask': (4, 9)}
